# NOTE

This file was used manually with each resulting file and the outputs transfered to the final poster. This is why you will not find the output files anywhere. Also, llm_framing_experiment.ipynb will need to be run to gain the actual results. I can send those results upon request.

# McNemar

In [129]:
# requirements:

import pandas as pd
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.contingency_tables import cochrans_q
from statsmodels.stats.multitest import multipletests
from pathlib import Path


In [130]:
# load results.csv (expects columns: question, condition, correct)
# 'correct' should be 0/1 (or bool). 'question' should uniquely identify the question.

# This path needs to be changed for each file you are analyzing
df = pd.read_csv("output/resultsSimpleMed.csv")

# If your 'question' column is full text and duplicates exist, ensure uniqueness by question index:
# If your dataset has an "idx" column (question id), prefer that:
if "question" in df.columns:
    id_col = "question"
else:
    # create a question id by grouping unique question text
    df["qtext_id"] = df["question"].astype(str)
    df["idx"] = df["qtext_id"].factorize()[0]
    id_col = "idx"

# Pivot to wide: one row per question, columns control/low/high with correct=0/1
df_wide = df.pivot_table(index=id_col, columns="condition", values="correct", aggfunc="max")

# Ensure columns exist
for c in ["control", "low", "high"]:
    if c not in df_wide.columns:
        df_wide[c] = np.nan

# Drop rows that are missing either of the two conditions for a pairwise test
# (You can decide whether to keep only rows where all three exist.)
df_wide = df_wide.reset_index()


In [131]:

# Define helper to run McNemar and report results
def run_mcnemar_pair(dfw, colA, colB, exact=True):
    sub = dfw[[colA, colB]].dropna()
    # ensure binary 0/1
    sub = sub[(sub[colA].isin([0,1])) & (sub[colB].isin([0,1]))]

    # build contingency counts:
    #        B=1   B=0
    # A=1    n11   n10
    # A=0    n01   n00
    n11 = int(((sub[colA]==1) & (sub[colB]==1)).sum())
    n10 = int(((sub[colA]==1) & (sub[colB]==0)).sum())
    n01 = int(((sub[colA]==0) & (sub[colB]==1)).sum())
    n00 = int(((sub[colA]==0) & (sub[colB]==0)).sum())

    table = [[n11, n10],
             [n01, n00]]
    # run McNemar
    # exact=True uses exact binomial test; exact=False uses chi-square approximation with continuity correction
    result = mcnemar(table, exact=exact)
    # paired difference in proportions: proportion that improved minus proportion that worsened
    # proportion that became correct in B but wrong in A = n01 / N
    # proportion that became wrong in B but correct in A = n10 / N
    N = n11 + n10 + n01 + n00
    prop_B_better = n01 / N
    prop_A_better = n10 / N
    prop_diff = prop_B_better - prop_A_better

    return {
        "pair": f"{colA} vs {colB}",
        "N_pairs": N,
        "n11": n11, "n10": n10, "n01": n01, "n00": n00,
        "prop_B_better": prop_B_better,
        "prop_A_better": prop_A_better,
        "prop_diff (B - A)": prop_diff,
        "statistic": result.statistic,
        "pvalue": result.pvalue#,
       # "method": "exact" if result.method == "exact" else "asymptotic"
    }


In [132]:

# Run the two planned comparisons (control vs low) and (control vs high)
pairs = [("control", "low"), ("control", "high")]
results = []
for a,b in pairs:
    res = run_mcnemar_pair(df_wide, a, b, exact=True)
    results.append(res)

res_df = pd.DataFrame(results)
print(res_df[["pair","N_pairs","n10","n01","prop_diff (B - A)","statistic","pvalue"]])

# Multiple comparisons correction (2 tests)
pvals = res_df["pvalue"].values
# choose method: 'bonferroni' or 'holm'
rej_bonf, p_adj_bonf, _, _ = multipletests(pvals, alpha=0.05, method="bonferroni")
rej_holm, p_adj_holm, _, _ = multipletests(pvals, alpha=0.05, method="holm")

res_df["p_adj_bonf"] = p_adj_bonf
res_df["reject_bonf"] = rej_bonf
res_df["p_adj_holm"] = p_adj_holm
res_df["reject_holm"] = rej_holm

print("\nPairwise McNemar results (with multiple-comparison correction):")
print(res_df[["pair","N_pairs","n10","n01","prop_diff (B - A)","pvalue","p_adj_bonf","reject_bonf","p_adj_holm","reject_holm"]])


              pair  N_pairs  n10  n01  prop_diff (B - A)  statistic  pvalue
0   control vs low      200    2    1             -0.005        1.0     1.0
1  control vs high      200    3    3              0.000        3.0     1.0

Pairwise McNemar results (with multiple-comparison correction):
              pair  N_pairs  n10  n01  prop_diff (B - A)  pvalue  p_adj_bonf  \
0   control vs low      200    2    1             -0.005     1.0         1.0   
1  control vs high      200    3    3              0.000     1.0         1.0   

   reject_bonf  p_adj_holm  reject_holm  
0        False         1.0        False  
1        False         1.0        False  


## Cohcrane's Q

In [134]:
def run_cochran_q(df):
    """Runs Cochran's Q test using control/low/high."""
    k = ["control", "low", "high"]

    # Make sure values are integers 0/1
    data = df[k].astype(int).values  # shape: (N_questions, 3)

    # Run Cochran's Q
    result = cochrans_q(data)

    return {
        "Q statistic": result.statistic,
        "p-value": result.pvalue,
        "N questions": len(df)
    }

In [135]:
q_results = run_cochran_q(df_wide)
print("Cochran's Q results:")
print(q_results)

Cochran's Q results:
{'Q statistic': 0.2857142857142857, 'p-value': 0.8668778997501817, 'N questions': 200}
